# Run an external robot pose estimator

Runs an external robot pose estimator (RoboPEPP or HPE, see `DESIGN-Import-Competitors.md`) on the demonstrations of an exp/run of the family `external_proprioception`:

1. exports the frames of the demonstrations of `validation_data` as a DREAM-format folder (links to the original frames of their source sequence);
2. runs the driver of the method in the method's own Python environment, which writes the predicted joint angles of every frame to `angles_validation_data.npz` in the run's directory.

The predictions are then read back by `predict_positions()`, so `Verify_VisualProprioception.ipynb` and `Compare_VisualProprioception.ipynb` evaluate an external run like one of our models.

In [ ]:
import sys
sys.path.append("..")
from exp_run_config import Config
Config.PROJECTNAME = "BerryPicker"

import pathlib
from pprint import pprint
import numpy as np

from visual_proprioception import external_proprioception
from visual_proprioception.visproprio_helper import predict_positions

## Parameters

In [ ]:
# *** Initialize the variables with default values
# *** This cell should be tagged as parameters
# *** If papermill is used, some of the values will be overwritten

# If it is set to discard-old, the exprun will be recreated from scratch
creation_style = "exist-ok"

# If not None, set an external experiment path
expruns_path = None
# If not None, set an output path
results_path = None

experiment = "external_proprioception"
run = "robopepp_panda_published"

In [ ]:
if expruns_path:
    expruns_path = pathlib.Path(expruns_path)
    assert expruns_path.exists()
    Config().set_exprun_path(expruns_path)
    Config().copy_experiment("external_proprioception")
    Config().copy_experiment("demonstration")
if results_path:
    results_path = pathlib.Path(results_path)
    assert results_path.exists()
    Config().set_results_path(results_path)

exp = Config().get_experiment(experiment, run, creation_style=creation_style)
pprint(exp)

# the robot of the run (franka, ...): its exp/runs go to the workspace too
if expruns_path:
    Config().copy_experiment(exp["robot_exp"])
exp_robot = Config().get_experiment(exp["robot_exp"], exp["robot_run"])

## Export the frames and run the method

In [ ]:
external_proprioception.export_dream(exp, "validation_data")
exp.start_timer("external-inference")
external_proprioception.run_external(exp, "validation_data")
exp.end_timer("external-inference")

## A first look at the predictions

The per-field RMSE in normalized units, as Verify and Compare report it.

In [ ]:
from robot.position_factory import position_fields

result = predict_positions(exp, exp_robot, "validation_data")
for i, field in enumerate(position_fields(exp_robot)):
    rmse = np.sqrt(np.mean((result["predictions"][:, i] - result["targets"][:, i]) ** 2))
    print(f"{field:<16}{rmse:.4f}")

In [ ]:
exp.done()